In [18]:
import pandas as pd
import numpy as np
import pyfixest as pf

In [19]:
labour_df = pd.read_csv('/Users/W.Zhouhan/GitHub/EC2C3/pset4/labour_data.csv')
labour_df.dtypes

year                            int64
month                           int64
age                             int64
num_children_under_18           int64
state                             str
employed                        int64
hours_worked_week               int64
female                          int64
educ                              str
race                              str
marital_status                    str
household_income                  str
industry                          str
commute_reason                    str
commuting_expectation_of_job      str
dtype: object

In [20]:
labour_df.shape

(34910, 15)

In [21]:
labour_df.isna().sum()

year                                0
month                               0
age                                 0
num_children_under_18               0
state                               0
employed                            0
hours_worked_week                   0
female                              0
educ                               13
race                                0
marital_status                     15
household_income                    0
industry                          208
commute_reason                  25290
commuting_expectation_of_job    23245
dtype: int64

In [22]:
int_columns = ['year', 'month', 'age', 'num_children_under_18', 'employed', 'hours_worked_week', 'female']
labour_df[int_columns].describe()

,year,month,age,num_children_under_18,employed,hours_worked_week,female
count,34910.000000,34910.000000,34910.000000,34910.000000,34910.0,34910.000000,34910.000000
mean,2020.306646,6.458178,34.020796,0.864967,1.0,37.631080,0.468233
std,0.461108,2.889311,8.830624,1.028615,0.0,16.803481,0.507083
min,2020.000000,1.000000,18.000000,0.000000,1.0,0.000000,0.000000
25%,2020.000000,4.000000,27.000000,0.000000,1.0,30.000000,0.000000
50%,2020.000000,6.000000,34.000000,0.000000,1.0,40.000000,0.000000
75%,2021.000000,9.000000,41.000000,2.000000,1.0,45.000000,1.000000
max,2021.000000,12.000000,50.000000,3.000000,1.0,80.000000,2.000000


In [23]:
# drop female = 2.0
o = len(labour_df)
labour_df = labour_df.loc[labour_df['female'] <= 1.0]
n = len(labour_df)
print(f'Rows dropped: {o-n}')

Rows dropped: 142


In [25]:
# Regress hours on children
help(pf.feols)
fit_children = pf.feols('hours_worked_week ~ num_children_under_18', data=labour_df, vcov='hetero')

Help on function feols in module pyfixest.estimation.api.feols:

feols(
    fml: str,
    data: DataFrameType,
    vcov: VcovTypeOptions | dict[str, str] | None = None,
    vcov_kwargs: dict[str, str | int] | None = None,
    weights: None | str = None,
    ssc: dict[str, str | bool] | None = None,
    fixef_rm: FixedRmOptions = 'singleton',
    collin_tol: float = 1e-09,
    drop_intercept: bool = False,
    copy_data: bool = True,
    store_data: bool = True,
    lean: bool = False,
    weights_type: WeightsTypeOptions = 'aweights',
    solver: SolverOptions = 'scipy.linalg.solve',
    demeaner: AnyDemeaner | None = None,
    demeaner_backend: str | None = None,
    fixef_tol: float | None = None,
    fixef_maxiter: int | None = None,
    use_compression: bool = False,
    reps: int = 100,
    context: int | Mapping[str, Any] | None = None,
    seed: int | None = None,
    split: str | None = None,
    fsplit: str | None = None
) -> Feols | FixestMulti
    Estimate a linear regressio

In [26]:
fit_children.summary()

###

Estimation:  OLS
Dep. var.: hours_worked_week
sample: None = all
Inference:  hetero
Observations:  34768

| Coefficient           |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept             |     37.718 |        0.114 |   332.304 |      0.000 | 37.496 |  37.941 |
| num_children_under_18 |     -0.091 |        0.089 |    -1.023 |      0.306 | -0.264 |   0.083 |
---
RMSE: 16.787 R2: 0.0 


In [28]:
labour_df['marital_status'].value_counts(dropna=False)

marital_status
Married          20137
Never Married    11991
Divorced          1533
Separated          672
Widowed            420
NaN                 15
Name: count, dtype: int64

In [29]:
labour_df['married'] = (labour_df['marital_status'] == 'Married').astype(int)

In [30]:
fit_long = pf.feols('hours_worked_week ~ num_children_under_18 + married', data=labour_df, vcov='hetero')

In [31]:
fit_long.summary()

###

Estimation:  OLS
Dep. var.: hours_worked_week
sample: None = all
Inference:  hetero
Observations:  34768

| Coefficient           |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept             |     35.952 |        0.148 |   242.415 |      0.000 | 35.661 |  36.243 |
| num_children_under_18 |     -0.581 |        0.090 |    -6.425 |      0.000 | -0.758 |  -0.404 |
| married               |      3.783 |        0.187 |    20.201 |      0.000 |  3.416 |   4.150 |
---
RMSE: 16.691 R2: 0.012 


In [32]:
# estimate auxiliary equation to check answer
fit_aux = pf.feols('married ~ num_children_under_18', data=labour_df, vcov='hetero')
fit_aux.summary()

###

Estimation:  OLS
Dep. var.: married
sample: None = all
Inference:  hetero
Observations:  34768

| Coefficient           |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept             |      0.467 |        0.003 |   134.582 |      0.000 |  0.460 |   0.474 |
| num_children_under_18 |      0.130 |        0.002 |    54.328 |      0.000 |  0.125 |   0.134 |
---
RMSE: 0.475 R2: 0.073 


In [33]:
# investigate residuals for long and short regressions
resid_children = fit_children.resid()
resid_long = fit_long.resid()

# mean and variance
print(resid_children.mean(), resid_children.var())
print(resid_long.mean(), resid_long.var())


9.534940715492187e-15 281.81590733895956
9.861927708478887e-15 278.58318086880286


In [34]:
# standardise hours for short regression
fit_children_std_baseline = pf.feols(
    'standardize(hours_worked_week) ~ num_children_under_18',
    data=labour_df
)
fit_children_std_baseline.summary()

###

Estimation:  OLS
Dep. var.: standardize(hours_worked_week)
sample: None = all
Inference:  iid
Observations:  34768

| Coefficient           |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept             |      0.005 |        0.007 |     0.667 |      0.505 | -0.009 |   0.018 |
| num_children_under_18 |     -0.005 |        0.005 |    -1.035 |      0.301 | -0.016 |   0.005 |
---
RMSE: 1.0 R2: 0.0 


In [35]:

fit_children_std_robust = pf.feols(
    'standardize(hours_worked_week) ~ num_children_under_18',
    data=labour_df, vcov='hetero'
)
fit_children_std_robust.summary()


###

Estimation:  OLS
Dep. var.: standardize(hours_worked_week)
sample: None = all
Inference:  hetero
Observations:  34768

| Coefficient           |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept             |      0.005 |        0.007 |     0.691 |      0.490 | -0.009 |   0.018 |
| num_children_under_18 |     -0.005 |        0.005 |    -1.023 |      0.306 | -0.016 |   0.005 |
---
RMSE: 1.0 R2: 0.0 
